# n = 5 agents per side

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
from req_functions_from_ApproxSP import *
from tqdm import tqdm
from generate_profiles import *
from euclidean import *
from mallows import *
from AccompliceManipulation import *
from impartial import *
import pandas as pd
from pathlib import Path
Path("odd_market_robustness_results").mkdir(exist_ok=True)

**Mallows MA**

In [ ]:

def generate_MA(num_agents):
    votes_left = []
    votes_right = []

    for i in range(num_agents):
        row_left = [(i + j) % num_agents for j in range(num_agents)]
        votes_left.append(row_left)

        row_right = [(i - j) % num_agents for j in range(num_agents)]
        votes_right.append(row_right)

    return votes_left, votes_right

def generate_mallows_MA_votes(num_agents, phi_m, phi_w, **kwargs):
    """ Mallows on top of Asymmetric instance """
    
    votes_left_array, votes_right_array = generate_MA(num_agents)
    
    # Apply Mallows
    votes_left_array = mallows_votes(votes_left_array, phi_m)
    votes_right_array = mallows_votes(votes_right_array, phi_w)
    
    # No need for .tolist()
    votes_left = votes_left_array
    votes_right = votes_right_array

 
    j = 0
    for row in votes_left:
        for i in range(len(row)):
            votes_left[j][i] += 1
        j+=1
    j = 0
    for row in votes_right:
        for i in range(len(row)):
            votes_right[j][i] += 1
        j += 1
 
 
    return [votes_left, votes_right]
def mallows_vote(vote, phi):
    num_candidates = len(vote)
    raw_vote = generate_mallows_votes(1, num_candidates, phi=phi, weight=0)[0]
    new_vote = [0] * len(vote)
    for i in range(num_candidates):
        new_vote[raw_vote[i]] = vote[i]
    return new_vote
 
 
def mallows_votes(votes, phi):
    for i in range(len(votes)):
        votes[i] = mallows_vote(votes[i], phi)
    return votes
 
def generate_mallows_votes(num_voters, num_candidates, phi=0.5, weight=0, **kwargs):
    if phi is None:
        logging.warning('phi is not defined')
    insertion_probabilites_list = []
    for i in range(1, num_candidates):
        insertion_probabilites_list.append(computeInsertionProbas(i, phi))
    V = []
    for i in range(num_voters):
        vote = mallowsVote(num_candidates, insertion_probabilites_list)
        if weight > 0:
            probability = np.random.random()
            if probability <= weight:
                vote.reverse()
        V += [vote]
    return V
 

def computeInsertionProbas(i, phi):
    probas = (i + 1) * [0]
    for j in range(i + 1):
        probas[j] = pow(phi, (i + 1) - (j + 1))
    return probas


def mallowsVote(m, insertion_probabilites_list):
    vote = [0]
    for i in range(1, m):
        index = weighted_choice(insertion_probabilites_list[i - 1])
        vote.insert(index, i)
    return vote


def weighted_choice(choices):
    total = 0
    for w in choices:
        total = total + w
    r = np.random.uniform(0, total)
    upto = 0.0
    for i, w in enumerate(choices):
        if upto + w >= r:
            return i
        upto = upto + w
    assert False, "Shouldn'process_id get here"

In [3]:
##Mallows MA

data_MA = []
numberMA = []
numberMA_pairs = []
n=5
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_MA_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MA.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMA.append(number_phi)
    numberMA_pairs.append(number_phi_pairs) 
print(numberMA)
print(numberMA_pairs) 

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.12, 0.08, 0.12, 0.16, 0.28], [0.14, 0.08, 0.22, 0.08, 0.18, 0.32], [0.48, 0.36, 0.1, 0.3, 0.24, 0.28], [0.56, 0.28, 0.28, 0.16, 0.38, 0.18], [0.58, 0.48, 0.46, 0.44, 0.2, 0.14]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.18, 0.08, 0.2, 0.22, 0.38], [0.14, 0.12, 0.32, 0.16, 0.22, 0.44], [0.58, 0.46, 0.1, 0.38, 0.24, 0.56], [0.62, 0.44, 0.32, 0.28, 0.54, 0.38], [0.74, 0.66, 0.58, 0.6, 0.28, 0.14]]


In [4]:
df2 = pd.DataFrame(data_MA, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('odd_market_robustness_results/5_mallows_MA.csv', index=False)

In [5]:
df = pd.read_csv('odd_market_robustness_results/5_mallows_MA.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/5_mallows_MA.csv', index=False)

**Mallows Euclidean**

In [6]:
## Euclidean Mallows Accomplice for d = 2

data_EM = []
numberEM = []
numberEM_pairs = []
n=5
phi_list=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count=[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_euclidean_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt=0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_EM.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberEM.append(number_phi)
    numberEM_pairs.append(number_phi_pairs)
print(numberEM)
print(numberEM_pairs)

[[0.0, 0.0, 0.12, 0.12, 0.24, 0.24], [0.0, 0.0, 0.12, 0.12, 0.24, 0.12], [0.12, 0.1, 0.2, 0.2, 0.2, 0.24], [0.22, 0.16, 0.2, 0.14, 0.2, 0.2], [0.06, 0.16, 0.22, 0.36, 0.36, 0.3], [0.24, 0.18, 0.08, 0.32, 0.34, 0.08]]
[[0.0, 0.0, 0.16, 0.2, 0.24, 0.32], [0.0, 0.0, 0.16, 0.16, 0.28, 0.2], [0.2, 0.14, 0.36, 0.2, 0.32, 0.4], [0.3, 0.16, 0.24, 0.14, 0.2, 0.2], [0.12, 0.2, 0.22, 0.36, 0.52, 0.34], [0.28, 0.3, 0.08, 0.48, 0.38, 0.08]]


In [7]:
df2 = pd.DataFrame(data_EM, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('odd_market_robustness_results/5_euclidean_mallows.csv', index=False)

In [8]:
df = pd.read_csv('odd_market_robustness_results/5_euclidean_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/5_euclidean_mallows.csv', index=False)

**Reverse Euclidean**

In [9]:
#Reverse Euclidean

number_reverse_euclidean = []
number_reverse_euclidean_pairs = []
data_RE = []
n=5
p_list = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list  = [2, 4, 6, 8, 10]
for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for p in tqdm(p_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_reverse_euclidean_votes(n, d, 'uniform', 0.5, p)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_RE.append([d, p, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_reverse_euclidean.append(number_d)
    number_reverse_euclidean_pairs.append(number_d_pairs)
print(number_reverse_euclidean)
print(number_reverse_euclidean_pairs)

[[0.0, 0.04, 0.26, 0.26, 0.0, 0.0], [0.0, 0.12, 0.16, 0.4, 0.0, 0.0], [0.0, 0.14, 0.16, 0.12, 0.0, 0.0], [0.0, 0.2, 0.22, 0.3, 0.0, 0.0], [0.0, 0.12, 0.22, 0.4, 0.0, 0.0]]
[[0.0, 0.04, 0.34, 0.26, 0.0, 0.0], [0.0, 0.12, 0.26, 0.44, 0.0, 0.0], [0.0, 0.22, 0.2, 0.12, 0.0, 0.0], [0.0, 0.28, 0.3, 0.44, 0.0, 0.0], [0.0, 0.2, 0.34, 0.68, 0.0, 0.0]]


In [10]:
df3 = pd.DataFrame(data_RE, columns=["dimension", "proportion", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df3.to_csv('odd_market_robustness_results/5_reverse_euclidean.csv', index=False)

In [11]:
df = pd.read_csv('odd_market_robustness_results/5_reverse_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/5_reverse_euclidean.csv', index=False)

**Mallows MD**

In [12]:
##Mallows MD

data_MD = []
numberMD = []
numberMD_pairs = []
n=5
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_asymmetric_votes(num_agents=n, phi_m=phi1, phi_w=phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MD.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMD.append(number_phi)
    numberMD_pairs.append(number_phi_pairs) 
print(numberMD)
print(numberMD_pairs) 

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.36, 1.3, 0.7, 0.54, 0.3, 0.46], [1.34, 0.7, 0.54, 0.56, 0.44, 0.28], [1.2, 0.8, 0.54, 0.74, 0.44, 0.1], [0.84, 0.74, 0.6, 0.44, 0.26, 0.34], [0.44, 0.7, 0.5, 0.12, 0.62, 0.04]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.52, 1.58, 0.76, 0.68, 0.34, 0.6], [2.02, 0.88, 0.88, 0.68, 0.58, 0.4], [1.74, 1.04, 0.82, 0.86, 0.56, 0.1], [1.16, 0.94, 0.88, 0.64, 0.38, 0.54], [0.7, 1.1, 0.62, 0.12, 0.9, 0.04]]


In [13]:
df4 = pd.DataFrame(data_MD, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df4.to_csv('odd_market_robustness_results/5_mallows_MD.csv', index=False)

In [14]:
df = pd.read_csv('odd_market_robustness_results/5_mallows_MD.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/5_mallows_MD.csv', index=False)

**Mallows**

In [15]:
##Mallows

numberM = []
numberM_pairs = []
data_M = []
n=5
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count =[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            m_pref = [generate_ic_votes(n)[0]]*n
            w_pref = [generate_ic_votes(n)[0]]*n
            mpref= mallows_votes(m_pref, phi1)
            wpref= mallows_votes(w_pref, phi2)
            mpref_clean = [[int(x) for x in row] for row in mpref] #these two are for saving purposes
            wpref_clean = [[int(x) for x in row] for row in wpref]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice) 
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_M.append([phi1, phi2, json.dumps(mpref_clean), json.dumps(wpref_clean), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberM.append(number_phi)
    numberM_pairs.append(number_phi_pairs)
print(numberM)
print(numberM_pairs)

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.14, 0.0, 0.06], [0.0, 0.04, 0.1, 0.08, 0.3, 0.2], [0.0, 0.16, 0.06, 0.2, 0.22, 0.24], [0.0, 0.0, 0.04, 0.28, 0.38, 0.16], [0.0, 0.04, 0.14, 0.18, 0.16, 0.34]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.14, 0.0, 0.06], [0.0, 0.08, 0.16, 0.08, 0.44, 0.24], [0.0, 0.16, 0.06, 0.28, 0.26, 0.24], [0.0, 0.0, 0.04, 0.36, 0.46, 0.24], [0.0, 0.04, 0.26, 0.22, 0.2, 0.42]]


In [16]:
df5 = pd.DataFrame(data_M, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df5.to_csv('odd_market_robustness_results/5_mallows.csv', index=False)

In [17]:
df = pd.read_csv('odd_market_robustness_results/5_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/5_mallows.csv', index=False)

**Expectations Euclidean**

In [18]:
import numpy as np

def generate_expectation_votes(num_agents: int = None,
                               dim: int = None,
                               space='uniform',
                               std: float = None,
                               phi=0.5,
                               **kwargs):
    name = f'{dim}d_{space}'

    left_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    left_agents_wishes = np.zeros([num_agents, dim])

    right_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    right_agents_wishes = np.zeros([num_agents, dim])

    for v in range(num_agents):
        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        left_agents_wishes[v][0] = np.random.normal(left_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        left_agents_wishes[v][1] = np.random.normal(left_agents_reality[v][1], std)

        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        right_agents_wishes[v][0] = np.random.normal(right_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        right_agents_wishes[v][1] = np.random.normal(right_agents_reality[v][1], std)

    left_votes = np.zeros([num_agents, num_agents], dtype=int)
    right_votes = np.zeros([num_agents, num_agents], dtype=int)
    distances = np.zeros([num_agents, num_agents], dtype=float)

    for v in range(num_agents):
        for c in range(num_agents):
            left_votes[v][c] = c
            distances[v][c] = np.linalg.norm(right_agents_reality[c] - left_agents_wishes[v])
        left_votes[v] = [x for _, x in sorted(zip(distances[v], left_votes[v]))]

    for v in range(num_agents):
        for c in range(num_agents):
            right_votes[v][c] = c
            distances[v][c] = np.linalg.norm(left_agents_reality[c] - right_agents_wishes[v])
        right_votes[v] = [x for _, x in sorted(zip(distances[v], right_votes[v]))]

    left_votes_lst = [list(row) for row in left_votes]
    right_votes_lst = [list(row) for row in right_votes]


    for i in range(len(left_votes_lst)):
        for j in range(len(left_votes_lst[i])):
            left_votes_lst[i][j] = left_votes_lst[i][j] + 1
    for i in range(len(right_votes_lst)):
        for j in range(len(right_votes_lst[i])):
            right_votes_lst[i][j] = right_votes_lst[i][j] + 1

    return [left_votes_lst, right_votes_lst]

In [19]:
import numpy as np

def convert_to_builtin(obj):
    if isinstance(obj, dict):
        return {convert_to_builtin(k): convert_to_builtin(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_builtin(elem) for elem in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

In [20]:
## Expectations Euclidean

data_EE = []
number_expectations_euclidean = []
number_EE_pairs = []
n = 5
std_list = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list = [2, 4, 6, 8, 10]

for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for std in tqdm(std_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_expectation_votes(n, d, 'uniform', std, 0.5)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_EE.append([d, std, json.dumps(convert_to_builtin(mpref)), json.dumps(convert_to_builtin(wpref)), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs)/ len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_expectations_euclidean.append(number_d)
    number_EE_pairs.append(number_d_pairs)
print(number_expectations_euclidean)
print(number_EE_pairs)

[[0.0, 0.08, 0.1, 0.08, 0.12, 0.18], [0.0, 0.04, 0.04, 0.04, 0.08, 0.1], [0.0, 0.0, 0.0, 0.04, 0.04, 0.04], [0.0, 0.0, 0.0, 0.08, 0.08, 0.1], [0.0, 0.0, 0.0, 0.0, 0.04, 0.04]]
[[0.0, 0.12, 0.1, 0.12, 0.2, 0.22], [0.0, 0.04, 0.04, 0.04, 0.08, 0.16], [0.0, 0.0, 0.0, 0.04, 0.04, 0.04], [0.0, 0.0, 0.0, 0.08, 0.08, 0.1], [0.0, 0.0, 0.0, 0.0, 0.04, 0.04]]


In [21]:
df6 = pd.DataFrame(data_EE, columns=["dimension", "std", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df6.to_csv('odd_market_robustness_results/5_expectations_euclidean.csv', index=False)

In [22]:
df = pd.read_csv('odd_market_robustness_results/5_expectations_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/5_expectations_euclidean.csv', index=False)

In [23]:
def invert_proposer_matches(proposer_matches):
    """
    proposer_matches: list of length n where index i (0-based) is proposer (i+1)
                      and value proposer_matches[i] is the partner id (1-based).
    Returns: list `inv` of length n that maps partner -> proposer (1-based ids),
             i.e., inv[partner-1] = proposer_id
    """
    n = len(proposer_matches)
    inv = [0] * n
    for proposer_idx, partner in enumerate(proposer_matches):
        if partner == 0:
            # unmatched / sentinel (shouldn't happen in full matching)
            continue
        inv[partner - 1] = proposer_idx + 1
    return inv


def rank_differences_women(mpref, wpref, debug=False):
    """
    Return list of length n: for each woman w (1..n),
    diff[w-1] = rank_w(men-opt partner for w) - rank_w(women-opt partner for w).
    This should be >= 0 for all w (woman-optimal is at least as good).
    """
    # men-optimal: list mapping man -> woman (1-based IDs)
    men_optimal = gale_shapley(mpref, wpref)

    # women-proposing: returns list mapping woman -> man (1-based IDs)
    women_proposer_matches = gale_shapley(wpref, mpref)

    # invert to get men -> woman mapping for the women-optimal matching
    women_optimal = invert_proposer_matches(women_proposer_matches)

    n = len(mpref)
    differences = []

    for w in range(1, n + 1):
        # partner of woman w in men-optimal matching (this returns the man id who is matched to w)
        partner_men_opt = get_w_partner(w, men_optimal)   # returns man id
        # partner of woman w in women-optimal matching (man id)
        partner_women_opt = get_w_partner(w, women_optimal)

        # rank of those partners in woman's preference list (1-based ranks)
        rank_men_opt = wpref[w - 1].index(partner_men_opt) + 1
        rank_women_opt = wpref[w - 1].index(partner_women_opt) + 1

        diff = rank_men_opt - rank_women_opt
        differences.append(diff)

        if debug and diff < 0:
            print(f"DEBUG NEGATIVE: woman={w}, rank_men_opt={rank_men_opt}, "
                  f"rank_women_opt={rank_women_opt}, man_opt={partner_men_opt}, woman_opt={partner_women_opt}")
            print(" wpref[w-1] =", wpref[w-1])
            print(" men_optimal (man->woman) =", men_optimal)
            print(" women_proposer_matches (woman->man) =", women_proposer_matches)
            print(" women_optimal (man->woman) =", women_optimal)

    return differences


In [24]:
import pandas as pd
import ast
import numpy as np

# --- Your helper functions (from before) ---
# gale_shapley, get_w_partner, get_m_partner, rank_differences_women, etc.

def parse_preferences(pref_str):
    """Convert the string representation of preferences to lists."""
    return ast.literal_eval(pref_str)

def compute_differences(pref_str):
    """Compute rank differences for women for a given preference string."""
    mpref, wpref = parse_preferences(pref_str)
    return rank_differences_women(mpref, wpref)

def process_csv(filepath):
    """Load CSV, compute rank differences and average, return updated df."""
    df = pd.read_csv(filepath)
    
    # Identify the preferences column (assuming it's the 3rd column)
    df["preferences"] = df["mpref"] + "," + df["wpref"]  # Combine mpref and wpref into a single string
    pref_col = "preferences"  # Assuming the new column contains the preferences
    
    # Compute rank differences
    df["rank_differences"] = df[pref_col].apply(compute_differences)
    
    # Compute average rank difference per row
    df["avg_rank_differences"] = df["rank_differences"].apply(lambda x: np.mean(x))
    
    return df

def count_positive(lst_str):
    try:
        lst = ast.literal_eval(lst_str)  # convert string -> list
        return sum(1 for x in lst if x > 0)
    except (ValueError, SyntaxError):
        return 0  # fallback if parsing fails
    
def egalitarian_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n + 1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
        cost += wpref[w-1].index(m)
    return cost

import numpy as np


def find_all_stable_original(malechoice, femalechoice, n):
    # Output list with all matchings
    res = []
    # Initialization
    unchanged = [True] * n
    marriage = [0] * n
    malecounter = [0] * n
    # Female choice transformation

    fc = np.zeros((n, n), dtype=int)
    for i in range(1, n):
        for j in range(0, n - 1):
            fc[i][femalechoice[i][j]] = j
        fc[i][0] = n
        malecounter[i] = 0
        marriage[i] = 0

    def get_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        return(malematches)

    def print_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        print(*malematches)

    def proposal(i, malec, marriage, c):
        if i < 0:
            success = True
        elif (i == 0) or malec[i] == (n - 1) or not (unchanged[i]):
            success = False
        else:
            c += 1
            j = malec[i]
            malec[i] = j + 1
            success = refusal(i, malechoice[i][j], malec, marriage, c)
        return success

    def refusal(i, j, malec, marriage, c):
        if fc[j][abs(marriage[j])] > fc[j][i]:
            k = marriage[j]
            marriage[j] = i
            success = proposal(k, malec, marriage, c)
        else:
            success = proposal(i, malec, marriage, c)
        return success

    def breakmariage(malecounter, marriage, i, n, count):
        malecounter_ori = malecounter.copy()
        marriage_ori = marriage.copy()
        marriage[malechoice[i][malecounter[i] - 1]] = -i
        success = proposal(i, malecounter, marriage, count)

        if not (success):
            unchanged[i] = False
            malecounter_ref = malecounter_ori.copy()
            marriage_ref = marriage_ori.copy()
            malecounter = malecounter_ref
            marriage = marriage_ref
            return malecounter, marriage

        #print_marriage(marriage)  # marriage[i] means the woman who is married to i man
        new_marriage = get_marriage(marriage)  # convert into female solution
        res.append(new_marriage)
        # establish the relationships between the original and the obtained stable matching

        for j in range(i, n):
            # if i==2 and j==2:
            #     print("This is it")
            malecounter, marriage = breakmariage(malecounter, marriage, j, n, count)

        for j in range((i + 1), n):
            unchanged[j] = True
        # return malecounter_ori, abs(marriage_ori)
        return malecounter_ori, marriage_ori

    count = 0
    for i in range(1, n):
        proposal(i, malecounter, marriage, count)
    #print_marriage(marriage)

    new_marriage = get_marriage(marriage)  # convert into female solution
    res.append(new_marriage)
    # establish the relationships between the original and the obtained stable matching

    M_optimal_marriage = marriage.copy()
    M_optimal_counter = malecounter.copy()
    for i in range(1, n - 1):
        breakmariage(M_optimal_counter.copy(), M_optimal_marriage.copy(), i, n, count)
    return res

def men_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
    return cost

def women_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += wpref[w-1].index(m)
    return cost

In [25]:
files = [
    "odd_market_robustness_results/5_euclidean_mallows.csv",
    "odd_market_robustness_results/5_expectations_euclidean.csv",
    "odd_market_robustness_results/5_mallows.csv",
    "odd_market_robustness_results/5_mallows_MA.csv",
    "odd_market_robustness_results/5_mallows_MD.csv",
    "odd_market_robustness_results/5_reverse_euclidean.csv"
]
for f in files:
    df_updated = process_csv(f)
    out_file = f.replace(".csv", "_with_rank_diff.csv")
    df_updated.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

rank_gain_files = [f.replace(".csv", "_with_rank_diff.csv") for f in files]

for f in rank_gain_files:
    df = pd.read_csv(f)
    df["trunc_count"] = df["rank_differences"].apply(count_positive)
    egalitarian_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = egalitarian_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        egalitarian_matches.append(best_match)
    df["egalitarian_match"] = egalitarian_matches
    men_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = men_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        men_matches.append(best_match)
    df["men_opt_match"] = men_matches

    women_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = women_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        women_matches.append(best_match)
    df["women_opt_match"] = women_matches


    cost_diffs = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))

        egal = row["egalitarian_match"]  # <- no ast.literal_eval
        egal_cost = egalitarian_cost(egal, m_pref, w_pref)

        men_opt = row["men_opt_match"]
        men_cost1 = men_cost(men_opt, m_pref, w_pref)

        women_opt = row["women_opt_match"]
        women_cost1 = women_cost(women_opt, m_pref, w_pref)

        delta = egal_cost - men_cost1 - women_cost1
        cost_diffs.append(delta)



    df["trunc_egal"] = cost_diffs 

    stability_counts = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))
        num_stable = len(stable)
        stability_counts.append(num_stable)


    df["stability_count"] = stability_counts

    df["trunc_prevalence"] = (df["stability_count"] > 1).astype(int)

    df.to_csv(f)

Saved: odd_market_robustness_results/5_euclidean_mallows_with_rank_diff.csv
Saved: odd_market_robustness_results/5_expectations_euclidean_with_rank_diff.csv
Saved: odd_market_robustness_results/5_mallows_with_rank_diff.csv
Saved: odd_market_robustness_results/5_mallows_MA_with_rank_diff.csv
Saved: odd_market_robustness_results/5_mallows_MD_with_rank_diff.csv
Saved: odd_market_robustness_results/5_reverse_euclidean_with_rank_diff.csv


# n = 7 agents per side

In [26]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
from req_functions_from_ApproxSP import *
from tqdm import tqdm
from generate_profiles import *
from euclidean import *
from mallows import *
from AccompliceManipulation import *
from impartial import *
import pandas as pd
from pathlib import Path
Path("odd_market_robustness_results").mkdir(exist_ok=True)

**Mallows MA**

In [27]:

def generate_MA(num_agents):
    votes_left = []
    votes_right = []

    for i in range(num_agents):
        row_left = [(i + j) % num_agents for j in range(num_agents)]
        votes_left.append(row_left)

        row_right = [(i - j) % num_agents for j in range(num_agents)]
        votes_right.append(row_right)

    return votes_left, votes_right

def generate_mallows_MA_votes(num_agents, phi_m, phi_w, **kwargs):
    """ Mallows on top of Asymmetric instance """
    
    votes_left_array, votes_right_array = generate_MA(num_agents)
    
    # Apply Mallows
    votes_left_array = mallows_votes(votes_left_array, phi_m)
    votes_right_array = mallows_votes(votes_right_array, phi_w)
    
    # No need for .tolist()
    votes_left = votes_left_array
    votes_right = votes_right_array

 
    j = 0
    for row in votes_left:
        for i in range(len(row)):
            votes_left[j][i] += 1
        j+=1
    j = 0
    for row in votes_right:
        for i in range(len(row)):
            votes_right[j][i] += 1
        j += 1
 
 
    return [votes_left, votes_right]
def mallows_vote(vote, phi):
    num_candidates = len(vote)
    raw_vote = generate_mallows_votes(1, num_candidates, phi=phi, weight=0)[0]
    new_vote = [0] * len(vote)
    for i in range(num_candidates):
        new_vote[raw_vote[i]] = vote[i]
    return new_vote
 
 
def mallows_votes(votes, phi):
    for i in range(len(votes)):
        votes[i] = mallows_vote(votes[i], phi)
    return votes
 
def generate_mallows_votes(num_voters, num_candidates, phi=0.5, weight=0, **kwargs):
    if phi is None:
        logging.warning('phi is not defined')
    insertion_probabilites_list = []
    for i in range(1, num_candidates):
        insertion_probabilites_list.append(computeInsertionProbas(i, phi))
    V = []
    for i in range(num_voters):
        vote = mallowsVote(num_candidates, insertion_probabilites_list)
        if weight > 0:
            probability = np.random.random()
            if probability <= weight:
                vote.reverse()
        V += [vote]
    return V
 

def computeInsertionProbas(i, phi):
    probas = (i + 1) * [0]
    for j in range(i + 1):
        probas[j] = pow(phi, (i + 1) - (j + 1))
    return probas


def mallowsVote(m, insertion_probabilites_list):
    vote = [0]
    for i in range(1, m):
        index = weighted_choice(insertion_probabilites_list[i - 1])
        vote.insert(index, i)
    return vote


def weighted_choice(choices):
    total = 0
    for w in choices:
        total = total + w
    r = np.random.uniform(0, total)
    upto = 0.0
    for i, w in enumerate(choices):
        if upto + w >= r:
            return i
        upto = upto + w
    assert False, "Shouldn'process_id get here"

In [28]:
##Mallows MA

data_MA = []
numberMA = []
numberMA_pairs = []
n=7
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_MA_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MA.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMA.append(number_phi)
    numberMA_pairs.append(number_phi_pairs) 
print(numberMA)
print(numberMA_pairs) 

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.04, 0.0, 0.0, 0.26, 0.34, 0.86], [0.12, 0.06, 0.0, 0.32, 0.68, 0.64], [0.52, 0.38, 0.3, 0.32, 0.72, 0.92], [0.92, 0.64, 0.22, 0.62, 0.54, 0.62], [0.86, 1.44, 0.52, 0.82, 0.9, 0.86]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.16, 0.0, 0.0, 0.38, 0.38, 0.98], [0.2, 0.06, 0.0, 0.42, 0.86, 1.14], [1.14, 0.72, 0.5, 0.42, 1.52, 1.66], [1.78, 1.44, 0.56, 1.04, 0.8, 1.16], [1.54, 2.92, 0.76, 1.38, 1.56, 1.56]]


In [29]:
df2 = pd.DataFrame(data_MA, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('odd_market_robustness_results/7_mallows_MA.csv', index=False)

In [30]:
df = pd.read_csv('odd_market_robustness_results/7_mallows_MA.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/7_mallows_MA.csv', index=False)

**Mallows Euclidean**

In [31]:
## Euclidean Mallows Accomplice for d = 2

data_EM = []
numberEM = []
numberEM_pairs = []
n=7
phi_list=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count=[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_euclidean_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt=0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_EM.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberEM.append(number_phi)
    numberEM_pairs.append(number_phi_pairs)
print(numberEM)
print(numberEM_pairs)

[[0.0, 0.04, 0.12, 0.2, 0.24, 0.78], [0.04, 0.04, 0.08, 0.2, 0.26, 0.52], [0.12, 0.14, 0.26, 0.1, 0.36, 0.5], [0.16, 0.18, 0.24, 0.22, 0.62, 0.8], [0.56, 0.42, 0.54, 0.8, 0.66, 0.64], [0.62, 0.5, 0.7, 0.7, 0.48, 0.7]]
[[0.0, 0.08, 0.12, 0.24, 0.56, 1.6], [0.08, 0.08, 0.12, 0.52, 0.54, 0.94], [0.24, 0.22, 0.54, 0.1, 0.66, 0.96], [0.2, 0.3, 0.36, 0.5, 1.4, 1.12], [0.68, 0.56, 0.64, 1.52, 1.06, 1.06], [1.32, 0.9, 1.32, 1.14, 0.82, 1.1]]


In [32]:
df2 = pd.DataFrame(data_EM, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('odd_market_robustness_results/7_euclidean_mallows.csv', index=False)

In [33]:
df = pd.read_csv('odd_market_robustness_results/7_euclidean_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/7_euclidean_mallows.csv', index=False)

**Reverse Euclidean**

In [34]:
#Reverse Euclidean

number_reverse_euclidean = []
number_reverse_euclidean_pairs = []
data_RE = []
n=7
p_list = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list  = [2, 4, 6, 8, 10]
for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for p in tqdm(p_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_reverse_euclidean_votes(n, d, 'uniform', 0.5, p)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_RE.append([d, p, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_reverse_euclidean.append(number_d)
    number_reverse_euclidean_pairs.append(number_d_pairs)
print(number_reverse_euclidean)
print(number_reverse_euclidean_pairs)

[[0.0, 0.3, 0.56, 0.44, 0.2, 0.0], [0.0, 0.38, 0.62, 0.4, 0.1, 0.0], [0.0, 0.46, 0.44, 0.24, 0.1, 0.0], [0.0, 0.22, 0.54, 0.72, 0.26, 0.0], [0.0, 0.4, 0.68, 0.66, 0.16, 0.0]]
[[0.0, 0.36, 1.04, 1.2, 0.52, 0.0], [0.0, 0.84, 1.22, 0.66, 0.1, 0.0], [0.0, 0.94, 0.62, 0.36, 0.1, 0.0], [0.0, 0.52, 1.02, 1.08, 0.32, 0.0], [0.0, 0.72, 1.1, 1.22, 0.2, 0.0]]


In [35]:
df3 = pd.DataFrame(data_RE, columns=["dimension", "proportion", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df3.to_csv('odd_market_robustness_results/7_reverse_euclidean.csv', index=False)

In [36]:
df = pd.read_csv('odd_market_robustness_results/7_reverse_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/7_reverse_euclidean.csv', index=False)

**Mallows MD**

In [37]:
##Mallows MD

data_MD = []
numberMD = []
numberMD_pairs = []
n=7
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_asymmetric_votes(num_agents=n, phi_m=phi1, phi_w=phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MD.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMD.append(number_phi)
    numberMD_pairs.append(number_phi_pairs) 
print(numberMD)
print(numberMD_pairs) 

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.14, 0.9, 1.22, 0.82, 0.96, 0.62], [1.92, 1.82, 1.58, 1.2, 0.6, 0.74], [1.78, 1.5, 1.2, 0.76, 0.94, 0.7], [1.52, 0.9, 0.9, 1.28, 0.8, 0.86], [1.14, 0.56, 0.84, 0.8, 0.64, 0.62]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.64, 0.9, 1.54, 1.04, 1.42, 1.0], [3.66, 2.66, 2.08, 2.02, 0.84, 1.16], [3.26, 2.44, 1.86, 1.14, 1.4, 1.22], [2.86, 1.68, 1.62, 2.3, 1.24, 1.58], [2.0, 0.88, 1.5, 1.2, 1.0, 1.1]]


In [38]:
df4 = pd.DataFrame(data_MD, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df4.to_csv('odd_market_robustness_results/7_mallows_MD.csv', index=False)

In [39]:
df = pd.read_csv('odd_market_robustness_results/7_mallows_MD.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/7_mallows_MD.csv', index=False)

**Mallows**

In [40]:
##Mallows

numberM = []
numberM_pairs = []
data_M = []
n=7
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count =[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            m_pref = [generate_ic_votes(n)[0]]*n
            w_pref = [generate_ic_votes(n)[0]]*n
            mpref= mallows_votes(m_pref, phi1)
            wpref= mallows_votes(w_pref, phi2)
            mpref_clean = [[int(x) for x in row] for row in mpref] #these two are for saving purposes
            wpref_clean = [[int(x) for x in row] for row in wpref]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice) 
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_M.append([phi1, phi2, json.dumps(mpref_clean), json.dumps(wpref_clean), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberM.append(number_phi)
    numberM_pairs.append(number_phi_pairs)
print(numberM)
print(numberM_pairs)

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.08, 0.04, 0.08, 0.08], [0.0, 0.06, 0.04, 0.08, 0.06, 0.08], [0.0, 0.2, 0.34, 0.32, 0.2, 0.44], [0.0, 0.08, 0.2, 0.52, 0.5, 0.38], [0.0, 0.2, 0.18, 0.5, 0.6, 0.44]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.08, 0.04, 0.24, 0.08], [0.0, 0.18, 0.08, 0.08, 0.12, 0.2], [0.0, 0.2, 0.52, 0.48, 0.28, 0.76], [0.0, 0.12, 0.2, 0.7, 0.9, 0.9], [0.0, 0.2, 0.34, 0.9, 1.06, 0.68]]


In [41]:
df5 = pd.DataFrame(data_M, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df5.to_csv('odd_market_robustness_results/7_mallows.csv', index=False)

In [42]:
df = pd.read_csv('odd_market_robustness_results/7_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/7_mallows.csv', index=False)

**Expectations Euclidean**

In [43]:
import numpy as np

def generate_expectation_votes(num_agents: int = None,
                               dim: int = None,
                               space='uniform',
                               std: float = None,
                               phi=0.5,
                               **kwargs):
    name = f'{dim}d_{space}'

    left_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    left_agents_wishes = np.zeros([num_agents, dim])

    right_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    right_agents_wishes = np.zeros([num_agents, dim])

    for v in range(num_agents):
        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        left_agents_wishes[v][0] = np.random.normal(left_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        left_agents_wishes[v][1] = np.random.normal(left_agents_reality[v][1], std)

        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        right_agents_wishes[v][0] = np.random.normal(right_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        right_agents_wishes[v][1] = np.random.normal(right_agents_reality[v][1], std)

    left_votes = np.zeros([num_agents, num_agents], dtype=int)
    right_votes = np.zeros([num_agents, num_agents], dtype=int)
    distances = np.zeros([num_agents, num_agents], dtype=float)

    for v in range(num_agents):
        for c in range(num_agents):
            left_votes[v][c] = c
            distances[v][c] = np.linalg.norm(right_agents_reality[c] - left_agents_wishes[v])
        left_votes[v] = [x for _, x in sorted(zip(distances[v], left_votes[v]))]

    for v in range(num_agents):
        for c in range(num_agents):
            right_votes[v][c] = c
            distances[v][c] = np.linalg.norm(left_agents_reality[c] - right_agents_wishes[v])
        right_votes[v] = [x for _, x in sorted(zip(distances[v], right_votes[v]))]

    left_votes_lst = [list(row) for row in left_votes]
    right_votes_lst = [list(row) for row in right_votes]


    for i in range(len(left_votes_lst)):
        for j in range(len(left_votes_lst[i])):
            left_votes_lst[i][j] = left_votes_lst[i][j] + 1
    for i in range(len(right_votes_lst)):
        for j in range(len(right_votes_lst[i])):
            right_votes_lst[i][j] = right_votes_lst[i][j] + 1

    return [left_votes_lst, right_votes_lst]

In [44]:
import numpy as np

def convert_to_builtin(obj):
    if isinstance(obj, dict):
        return {convert_to_builtin(k): convert_to_builtin(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_builtin(elem) for elem in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

In [45]:
## Expectations Euclidean

data_EE = []
number_expectations_euclidean = []
number_EE_pairs = []
n = 7
std_list = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list = [2, 4, 6, 8, 10]

for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for std in tqdm(std_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_expectation_votes(n, d, 'uniform', std, 0.5)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_EE.append([d, std, json.dumps(convert_to_builtin(mpref)), json.dumps(convert_to_builtin(wpref)), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs)/ len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_expectations_euclidean.append(number_d)
    number_EE_pairs.append(number_d_pairs)
print(number_expectations_euclidean)
print(number_EE_pairs)

[[0.0, 0.08, 0.24, 0.4, 0.18, 0.48], [0.0, 0.08, 0.0, 0.1, 0.06, 0.18], [0.0, 0.0, 0.04, 0.0, 0.12, 0.1], [0.0, 0.0, 0.0, 0.06, 0.0, 0.16], [0.0, 0.0, 0.04, 0.04, 0.04, 0.04]]
[[0.0, 0.12, 0.32, 0.68, 0.3, 0.84], [0.0, 0.08, 0.0, 0.1, 0.12, 0.24], [0.0, 0.0, 0.04, 0.0, 0.16, 0.1], [0.0, 0.0, 0.0, 0.06, 0.0, 0.24], [0.0, 0.0, 0.04, 0.08, 0.08, 0.04]]


In [46]:
df6 = pd.DataFrame(data_EE, columns=["dimension", "std", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df6.to_csv('odd_market_robustness_results/7_expectations_euclidean.csv', index=False)

In [47]:
df = pd.read_csv('odd_market_robustness_results/7_expectations_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/7_expectations_euclidean.csv', index=False)

In [48]:
def invert_proposer_matches(proposer_matches):
    """
    proposer_matches: list of length n where index i (0-based) is proposer (i+1)
                      and value proposer_matches[i] is the partner id (1-based).
    Returns: list `inv` of length n that maps partner -> proposer (1-based ids),
             i.e., inv[partner-1] = proposer_id
    """
    n = len(proposer_matches)
    inv = [0] * n
    for proposer_idx, partner in enumerate(proposer_matches):
        if partner == 0:
            # unmatched / sentinel (shouldn't happen in full matching)
            continue
        inv[partner - 1] = proposer_idx + 1
    return inv


def rank_differences_women(mpref, wpref, debug=False):
    """
    Return list of length n: for each woman w (1..n),
    diff[w-1] = rank_w(men-opt partner for w) - rank_w(women-opt partner for w).
    This should be >= 0 for all w (woman-optimal is at least as good).
    """
    # men-optimal: list mapping man -> woman (1-based IDs)
    men_optimal = gale_shapley(mpref, wpref)

    # women-proposing: returns list mapping woman -> man (1-based IDs)
    women_proposer_matches = gale_shapley(wpref, mpref)

    # invert to get men -> woman mapping for the women-optimal matching
    women_optimal = invert_proposer_matches(women_proposer_matches)

    n = len(mpref)
    differences = []

    for w in range(1, n + 1):
        # partner of woman w in men-optimal matching (this returns the man id who is matched to w)
        partner_men_opt = get_w_partner(w, men_optimal)   # returns man id
        # partner of woman w in women-optimal matching (man id)
        partner_women_opt = get_w_partner(w, women_optimal)

        # rank of those partners in woman's preference list (1-based ranks)
        rank_men_opt = wpref[w - 1].index(partner_men_opt) + 1
        rank_women_opt = wpref[w - 1].index(partner_women_opt) + 1

        diff = rank_men_opt - rank_women_opt
        differences.append(diff)

        if debug and diff < 0:
            print(f"DEBUG NEGATIVE: woman={w}, rank_men_opt={rank_men_opt}, "
                  f"rank_women_opt={rank_women_opt}, man_opt={partner_men_opt}, woman_opt={partner_women_opt}")
            print(" wpref[w-1] =", wpref[w-1])
            print(" men_optimal (man->woman) =", men_optimal)
            print(" women_proposer_matches (woman->man) =", women_proposer_matches)
            print(" women_optimal (man->woman) =", women_optimal)

    return differences


In [49]:
import pandas as pd
import ast
import numpy as np

# --- Your helper functions (from before) ---
# gale_shapley, get_w_partner, get_m_partner, rank_differences_women, etc.

def parse_preferences(pref_str):
    """Convert the string representation of preferences to lists."""
    return ast.literal_eval(pref_str)

def compute_differences(pref_str):
    """Compute rank differences for women for a given preference string."""
    mpref, wpref = parse_preferences(pref_str)
    return rank_differences_women(mpref, wpref)

def process_csv(filepath):
    """Load CSV, compute rank differences and average, return updated df."""
    df = pd.read_csv(filepath)
    
    # Identify the preferences column (assuming it's the 3rd column)
    df["preferences"] = df["mpref"] + "," + df["wpref"]  # Combine mpref and wpref into a single string
    pref_col = "preferences"  # Assuming the new column contains the preferences
    
    # Compute rank differences
    df["rank_differences"] = df[pref_col].apply(compute_differences)
    
    # Compute average rank difference per row
    df["avg_rank_differences"] = df["rank_differences"].apply(lambda x: np.mean(x))
    
    return df

def count_positive(lst_str):
    try:
        lst = ast.literal_eval(lst_str)  # convert string -> list
        return sum(1 for x in lst if x > 0)
    except (ValueError, SyntaxError):
        return 0  # fallback if parsing fails
    
def egalitarian_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n + 1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
        cost += wpref[w-1].index(m)
    return cost

import numpy as np


def find_all_stable_original(malechoice, femalechoice, n):
    # Output list with all matchings
    res = []
    # Initialization
    unchanged = [True] * n
    marriage = [0] * n
    malecounter = [0] * n
    # Female choice transformation

    fc = np.zeros((n, n), dtype=int)
    for i in range(1, n):
        for j in range(0, n - 1):
            fc[i][femalechoice[i][j]] = j
        fc[i][0] = n
        malecounter[i] = 0
        marriage[i] = 0

    def get_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        return(malematches)

    def print_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        print(*malematches)

    def proposal(i, malec, marriage, c):
        if i < 0:
            success = True
        elif (i == 0) or malec[i] == (n - 1) or not (unchanged[i]):
            success = False
        else:
            c += 1
            j = malec[i]
            malec[i] = j + 1
            success = refusal(i, malechoice[i][j], malec, marriage, c)
        return success

    def refusal(i, j, malec, marriage, c):
        if fc[j][abs(marriage[j])] > fc[j][i]:
            k = marriage[j]
            marriage[j] = i
            success = proposal(k, malec, marriage, c)
        else:
            success = proposal(i, malec, marriage, c)
        return success

    def breakmariage(malecounter, marriage, i, n, count):
        malecounter_ori = malecounter.copy()
        marriage_ori = marriage.copy()
        marriage[malechoice[i][malecounter[i] - 1]] = -i
        success = proposal(i, malecounter, marriage, count)

        if not (success):
            unchanged[i] = False
            malecounter_ref = malecounter_ori.copy()
            marriage_ref = marriage_ori.copy()
            malecounter = malecounter_ref
            marriage = marriage_ref
            return malecounter, marriage

        #print_marriage(marriage)  # marriage[i] means the woman who is married to i man
        new_marriage = get_marriage(marriage)  # convert into female solution
        res.append(new_marriage)
        # establish the relationships between the original and the obtained stable matching

        for j in range(i, n):
            # if i==2 and j==2:
            #     print("This is it")
            malecounter, marriage = breakmariage(malecounter, marriage, j, n, count)

        for j in range((i + 1), n):
            unchanged[j] = True
        # return malecounter_ori, abs(marriage_ori)
        return malecounter_ori, marriage_ori

    count = 0
    for i in range(1, n):
        proposal(i, malecounter, marriage, count)
    #print_marriage(marriage)

    new_marriage = get_marriage(marriage)  # convert into female solution
    res.append(new_marriage)
    # establish the relationships between the original and the obtained stable matching

    M_optimal_marriage = marriage.copy()
    M_optimal_counter = malecounter.copy()
    for i in range(1, n - 1):
        breakmariage(M_optimal_counter.copy(), M_optimal_marriage.copy(), i, n, count)
    return res

def men_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
    return cost

def women_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += wpref[w-1].index(m)
    return cost

In [50]:
files = [
    "odd_market_robustness_results/7_euclidean_mallows.csv",
    "odd_market_robustness_results/7_expectations_euclidean.csv",
    "odd_market_robustness_results/7_mallows.csv",
    "odd_market_robustness_results/7_mallows_MA.csv",
    "odd_market_robustness_results/7_mallows_MD.csv",
    "odd_market_robustness_results/7_reverse_euclidean.csv"
]
for f in files:
    df_updated = process_csv(f)
    out_file = f.replace(".csv", "_with_rank_diff.csv")
    df_updated.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

rank_gain_files = [f.replace(".csv", "_with_rank_diff.csv") for f in files]

for f in rank_gain_files:
    df = pd.read_csv(f)
    df["trunc_count"] = df["rank_differences"].apply(count_positive)
    egalitarian_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = egalitarian_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        egalitarian_matches.append(best_match)
    df["egalitarian_match"] = egalitarian_matches
    men_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = men_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        men_matches.append(best_match)
    df["men_opt_match"] = men_matches

    women_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = women_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        women_matches.append(best_match)
    df["women_opt_match"] = women_matches


    cost_diffs = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))

        egal = row["egalitarian_match"]  # <- no ast.literal_eval
        egal_cost = egalitarian_cost(egal, m_pref, w_pref)

        men_opt = row["men_opt_match"]
        men_cost1 = men_cost(men_opt, m_pref, w_pref)

        women_opt = row["women_opt_match"]
        women_cost1 = women_cost(women_opt, m_pref, w_pref)

        delta = egal_cost - men_cost1 - women_cost1
        cost_diffs.append(delta)



    df["trunc_egal"] = cost_diffs 

    stability_counts = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))
        num_stable = len(stable)
        stability_counts.append(num_stable)


    df["stability_count"] = stability_counts

    df["trunc_prevalence"] = (df["stability_count"] > 1).astype(int)

    df.to_csv(f)

Saved: odd_market_robustness_results/7_euclidean_mallows_with_rank_diff.csv
Saved: odd_market_robustness_results/7_expectations_euclidean_with_rank_diff.csv
Saved: odd_market_robustness_results/7_mallows_with_rank_diff.csv
Saved: odd_market_robustness_results/7_mallows_MA_with_rank_diff.csv
Saved: odd_market_robustness_results/7_mallows_MD_with_rank_diff.csv
Saved: odd_market_robustness_results/7_reverse_euclidean_with_rank_diff.csv


# n = 9 agents per side

In [51]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
from req_functions_from_ApproxSP import *
from tqdm import tqdm
from generate_profiles import *
from euclidean import *
from mallows import *
from AccompliceManipulation import *
from impartial import *
import pandas as pd
from pathlib import Path
Path("odd_market_robustness_results").mkdir(exist_ok=True)

**Mallows MA**

In [52]:

def generate_MA(num_agents):
    votes_left = []
    votes_right = []

    for i in range(num_agents):
        row_left = [(i + j) % num_agents for j in range(num_agents)]
        votes_left.append(row_left)

        row_right = [(i - j) % num_agents for j in range(num_agents)]
        votes_right.append(row_right)

    return votes_left, votes_right

def generate_mallows_MA_votes(num_agents, phi_m, phi_w, **kwargs):
    """ Mallows on top of Asymmetric instance """
    
    votes_left_array, votes_right_array = generate_MA(num_agents)
    
    # Apply Mallows
    votes_left_array = mallows_votes(votes_left_array, phi_m)
    votes_right_array = mallows_votes(votes_right_array, phi_w)
    
    # No need for .tolist()
    votes_left = votes_left_array
    votes_right = votes_right_array

 
    j = 0
    for row in votes_left:
        for i in range(len(row)):
            votes_left[j][i] += 1
        j+=1
    j = 0
    for row in votes_right:
        for i in range(len(row)):
            votes_right[j][i] += 1
        j += 1
 
 
    return [votes_left, votes_right]
def mallows_vote(vote, phi):
    num_candidates = len(vote)
    raw_vote = generate_mallows_votes(1, num_candidates, phi=phi, weight=0)[0]
    new_vote = [0] * len(vote)
    for i in range(num_candidates):
        new_vote[raw_vote[i]] = vote[i]
    return new_vote
 
 
def mallows_votes(votes, phi):
    for i in range(len(votes)):
        votes[i] = mallows_vote(votes[i], phi)
    return votes
 
def generate_mallows_votes(num_voters, num_candidates, phi=0.5, weight=0, **kwargs):
    if phi is None:
        logging.warning('phi is not defined')
    insertion_probabilites_list = []
    for i in range(1, num_candidates):
        insertion_probabilites_list.append(computeInsertionProbas(i, phi))
    V = []
    for i in range(num_voters):
        vote = mallowsVote(num_candidates, insertion_probabilites_list)
        if weight > 0:
            probability = np.random.random()
            if probability <= weight:
                vote.reverse()
        V += [vote]
    return V
 

def computeInsertionProbas(i, phi):
    probas = (i + 1) * [0]
    for j in range(i + 1):
        probas[j] = pow(phi, (i + 1) - (j + 1))
    return probas


def mallowsVote(m, insertion_probabilites_list):
    vote = [0]
    for i in range(1, m):
        index = weighted_choice(insertion_probabilites_list[i - 1])
        vote.insert(index, i)
    return vote


def weighted_choice(choices):
    total = 0
    for w in choices:
        total = total + w
    r = np.random.uniform(0, total)
    upto = 0.0
    for i, w in enumerate(choices):
        if upto + w >= r:
            return i
        upto = upto + w
    assert False, "Shouldn'process_id get here"

In [53]:
##Mallows MA

data_MA = []
numberMA = []
numberMA_pairs = []
n=9
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_MA_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MA.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMA.append(number_phi)
    numberMA_pairs.append(number_phi_pairs) 
print(numberMA)
print(numberMA_pairs) 

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.12, 0.66, 0.66], [0.22, 0.12, 0.24, 0.3, 0.8, 1.3], [0.36, 0.58, 0.28, 0.58, 0.6, 0.62], [0.94, 0.58, 1.04, 0.62, 1.04, 0.56], [1.2, 1.68, 1.32, 1.3, 1.28, 1.04]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.2, 0.9, 1.18], [0.3, 0.36, 0.32, 0.5, 1.84, 2.56], [0.36, 1.08, 0.56, 0.98, 1.46, 1.32], [2.44, 2.06, 2.44, 1.32, 2.46, 1.0], [2.42, 3.12, 2.58, 2.74, 2.36, 1.8]]


In [54]:
df2 = pd.DataFrame(data_MA, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('odd_market_robustness_results/9_mallows_MA.csv', index=False)

In [55]:
df = pd.read_csv('odd_market_robustness_results/9_mallows_MA.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/9_mallows_MA.csv', index=False)

**Mallows Euclidean**

In [56]:
## Euclidean Mallows Accomplice for d = 2

data_EM = []
numberEM = []
numberEM_pairs = []
n=9
phi_list=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count=[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_euclidean_votes(n, phi1, phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt=0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_EM.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberEM.append(number_phi)
    numberEM_pairs.append(number_phi_pairs)
print(numberEM)
print(numberEM_pairs)

[[0.0, 0.0, 0.12, 0.3, 0.8, 1.06], [0.0, 0.08, 0.08, 0.28, 0.52, 1.54], [0.08, 0.12, 0.24, 0.22, 0.78, 1.04], [0.2, 0.36, 0.34, 0.54, 0.68, 1.0], [0.68, 0.62, 0.86, 0.7, 0.72, 1.1], [1.28, 0.96, 0.76, 1.18, 1.2, 1.02]]
[[0.0, 0.0, 0.12, 0.54, 1.84, 2.26], [0.0, 0.08, 0.16, 0.44, 1.5, 3.48], [0.16, 0.36, 0.44, 0.36, 1.78, 2.38], [0.44, 0.52, 0.8, 1.06, 2.04, 2.14], [1.6, 1.16, 1.58, 1.2, 1.54, 1.84], [3.08, 1.86, 1.54, 2.54, 1.68, 1.54]]


In [57]:
df2 = pd.DataFrame(data_EM, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "count_pairs"])
df2.to_csv('odd_market_robustness_results/9_euclidean_mallows.csv', index=False)

In [58]:
df = pd.read_csv('odd_market_robustness_results/9_euclidean_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/9_euclidean_mallows.csv', index=False)

**Reverse Euclidean**

In [59]:
#Reverse Euclidean

number_reverse_euclidean = []
number_reverse_euclidean_pairs = []
data_RE = []
n=9
p_list = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list  = [2, 4, 6, 8, 10]
for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for p in tqdm(p_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_reverse_euclidean_votes(n, d, 'uniform', 0.5, p)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_RE.append([d, p, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_reverse_euclidean.append(number_d)
    number_reverse_euclidean_pairs.append(number_d_pairs)
print(number_reverse_euclidean)
print(number_reverse_euclidean_pairs)

[[0.0, 0.2, 0.68, 0.66, 0.2, 0.0], [0.0, 0.54, 0.9, 0.8, 0.3, 0.0], [0.0, 0.6, 0.72, 0.94, 0.38, 0.0], [0.0, 0.5, 0.98, 0.78, 0.2, 0.0], [0.0, 0.52, 1.06, 0.84, 0.54, 0.0]]
[[0.0, 0.4, 1.74, 1.36, 0.4, 0.0], [0.0, 1.3, 2.04, 2.06, 0.78, 0.0], [0.0, 1.76, 2.12, 2.38, 1.14, 0.0], [0.0, 0.74, 1.96, 1.96, 0.52, 0.0], [0.0, 0.98, 2.58, 1.8, 1.26, 0.0]]


In [60]:
df3 = pd.DataFrame(data_RE, columns=["dimension", "proportion", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df3.to_csv('odd_market_robustness_results/9_reverse_euclidean.csv', index=False)

In [61]:
df = pd.read_csv('odd_market_robustness_results/9_reverse_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/9_reverse_euclidean.csv', index=False)

**Mallows MD**

In [62]:
##Mallows MD

data_MD = []
numberMD = []
numberMD_pairs = []
n=9
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            pref=generate_mallows_asymmetric_votes(num_agents=n, phi_m=phi1, phi_w=phi2)
            mpref=pref[0]
            wpref=pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range(1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count_pairs.append(cnt_pairs)
            count.append(count_woman)
            data_MD.append([phi1, phi2, json.dumps(mpref), json.dumps(wpref), samples, count_woman, cnt_pairs])
        if len(count) > 0: 
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberMD.append(number_phi)
    numberMD_pairs.append(number_phi_pairs) 
print(numberMD)
print(numberMD_pairs) 

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.22, 1.36, 2.24, 1.82, 1.24, 1.66], [2.7, 1.68, 1.74, 1.52, 1.58, 1.08], [1.88, 2.26, 1.8, 1.76, 1.58, 1.0], [1.44, 1.54, 1.94, 1.78, 1.54, 1.14], [1.34, 1.3, 1.7, 0.78, 1.2, 1.34]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2.88, 2.44, 3.44, 2.9, 1.98, 3.04], [4.66, 2.66, 2.92, 3.24, 2.14, 2.24], [4.48, 4.14, 4.32, 3.28, 2.94, 1.8], [3.58, 2.9, 4.08, 2.92, 3.16, 2.34], [3.66, 3.04, 3.2, 1.38, 2.14, 2.5]]


In [63]:
df4 = pd.DataFrame(data_MD, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df4.to_csv('odd_market_robustness_results/9_mallows_MD.csv', index=False)

In [64]:
df = pd.read_csv('odd_market_robustness_results/9_mallows_MD.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/9_mallows_MD.csv', index=False)

**Mallows**

In [65]:
##Mallows

numberM = []
numberM_pairs = []
data_M = []
n=9
phi_list=[0, 0.2, 0.4, 0.6, 0.8, 1]
for phi1 in tqdm(phi_list):
    number_phi = []
    number_phi_pairs = []
    for phi2 in tqdm(phi_list):
        count =[]
        count_pairs = []
        ct=[]
        for samples in range (50):
            count_woman = 0
            cnt_pairs = 0
            m_pref = [generate_ic_votes(n)[0]]*n
            w_pref = [generate_ic_votes(n)[0]]*n
            mpref= mallows_votes(m_pref, phi1)
            wpref= mallows_votes(w_pref, phi2)
            mpref_clean = [[int(x) for x in row] for row in mpref] #these two are for saving purposes
            wpref_clean = [[int(x) for x in row] for row in wpref]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice) 
                    if result [0]!= matching:
                        cnt = cnt+1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_M.append([phi1, phi2, json.dumps(mpref_clean), json.dumps(wpref_clean), samples, count_woman, cnt_pairs])
        if len(count)>0:
            number_phi.append(sum(count) / len(count))
            number_phi_pairs.append(sum(count_pairs) / len(count_pairs))
        else:
            number_phi.append(0)
            number_phi_pairs.append(0)
    numberM.append(number_phi)
    numberM_pairs.append(number_phi_pairs)
print(numberM)
print(numberM_pairs)

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.04, 0.08, 0.08, 0.08], [0.0, 0.08, 0.08, 0.34, 0.44, 0.38], [0.0, 0.04, 0.04, 0.7, 0.34, 0.48], [0.0, 0.18, 0.38, 0.54, 0.78, 1.08], [0.0, 0.0, 0.36, 0.54, 0.88, 1.62]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.04, 0.08, 0.12, 0.16], [0.0, 0.08, 0.12, 0.5, 0.74, 0.54], [0.0, 0.04, 0.04, 1.04, 0.44, 0.86], [0.0, 0.18, 0.94, 0.98, 1.42, 1.86], [0.0, 0.0, 0.62, 1.06, 1.32, 2.76]]


In [66]:
df5 = pd.DataFrame(data_M, columns=["phi_m", "phi_w", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df5.to_csv('odd_market_robustness_results/9_mallows.csv', index=False)

In [67]:
df = pd.read_csv('odd_market_robustness_results/9_mallows.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/9_mallows.csv', index=False)

**Expectations Euclidean**

In [68]:
import numpy as np

def generate_expectation_votes(num_agents: int = None,
                               dim: int = None,
                               space='uniform',
                               std: float = None,
                               phi=0.5,
                               **kwargs):
    name = f'{dim}d_{space}'

    left_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    left_agents_wishes = np.zeros([num_agents, dim])

    right_agents_reality = np.array([get_uniform_point(dim) for _ in range(num_agents)])
    right_agents_wishes = np.zeros([num_agents, dim])

    for v in range(num_agents):
        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        left_agents_wishes[v][0] = np.random.normal(left_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        left_agents_wishes[v][1] = np.random.normal(left_agents_reality[v][1], std)

        # while agents_wishes[v][0] <= 0 or agents_wishes[v][0] >= 1:
        right_agents_wishes[v][0] = np.random.normal(right_agents_reality[v][0], std)
        # while agents_wishes[v][1] <= 0 or agents_wishes[v][1] >= 1:
        right_agents_wishes[v][1] = np.random.normal(right_agents_reality[v][1], std)

    left_votes = np.zeros([num_agents, num_agents], dtype=int)
    right_votes = np.zeros([num_agents, num_agents], dtype=int)
    distances = np.zeros([num_agents, num_agents], dtype=float)

    for v in range(num_agents):
        for c in range(num_agents):
            left_votes[v][c] = c
            distances[v][c] = np.linalg.norm(right_agents_reality[c] - left_agents_wishes[v])
        left_votes[v] = [x for _, x in sorted(zip(distances[v], left_votes[v]))]

    for v in range(num_agents):
        for c in range(num_agents):
            right_votes[v][c] = c
            distances[v][c] = np.linalg.norm(left_agents_reality[c] - right_agents_wishes[v])
        right_votes[v] = [x for _, x in sorted(zip(distances[v], right_votes[v]))]

    left_votes_lst = [list(row) for row in left_votes]
    right_votes_lst = [list(row) for row in right_votes]


    for i in range(len(left_votes_lst)):
        for j in range(len(left_votes_lst[i])):
            left_votes_lst[i][j] = left_votes_lst[i][j] + 1
    for i in range(len(right_votes_lst)):
        for j in range(len(right_votes_lst[i])):
            right_votes_lst[i][j] = right_votes_lst[i][j] + 1

    return [left_votes_lst, right_votes_lst]

In [69]:
import numpy as np

def convert_to_builtin(obj):
    if isinstance(obj, dict):
        return {convert_to_builtin(k): convert_to_builtin(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_builtin(elem) for elem in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

In [70]:
## Expectations Euclidean

data_EE = []
number_expectations_euclidean = []
number_EE_pairs = []
n = 9
std_list = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
d_list = [2, 4, 6, 8, 10]

for d in tqdm(d_list):
    number_d = []
    number_d_pairs = []
    for std in tqdm(std_list):
        count = []
        count_pairs = []
        ct = []
        for samples in range(50):
            count_woman = 0
            cnt_pairs = 0
            pref = generate_expectation_votes(n, d, 'uniform', std, 0.5)
            mpref = pref[0]
            wpref = pref[1]
            matching = gale_shapley(mpref, wpref)
            for manipulator in range (1, n+1):
                cnt = 0
                for accomplice in range (1, n+1):
                    result = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
                    if result[0] != matching:
                        cnt += 1
                if cnt >= 1:
                    count_woman += 1
                    cnt_pairs += cnt
            count.append(count_woman)
            count_pairs.append(cnt_pairs)
            data_EE.append([d, std, json.dumps(convert_to_builtin(mpref)), json.dumps(convert_to_builtin(wpref)), samples, count_woman, cnt_pairs])
        if len(count) > 0:
            number_d.append(sum(count)/len(count))
            number_d_pairs.append(sum(count_pairs)/ len(count_pairs))
        else:
            number_d.append(0)
            number_d_pairs.append(0)
    number_expectations_euclidean.append(number_d)
    number_EE_pairs.append(number_d_pairs)
print(number_expectations_euclidean)
print(number_EE_pairs)

[[0.0, 0.2, 0.6, 0.84, 0.98, 0.8], [0.0, 0.0, 0.28, 0.26, 0.26, 0.2], [0.0, 0.04, 0.04, 0.16, 0.14, 0.38], [0.0, 0.0, 0.04, 0.12, 0.12, 0.26], [0.0, 0.0, 0.0, 0.26, 0.16, 0.44]]
[[0.0, 0.32, 1.6, 1.62, 2.6, 1.32], [0.0, 0.0, 0.48, 0.54, 0.5, 0.6], [0.0, 0.04, 0.04, 0.24, 0.14, 0.58], [0.0, 0.0, 0.04, 0.12, 0.36, 0.46], [0.0, 0.0, 0.0, 0.3, 0.48, 0.8]]


In [71]:
df6 = pd.DataFrame(data_EE, columns=["dimension", "std", "mpref", "wpref", "sample", "count_woman", "cnt_pairs"])
df6.to_csv('odd_market_robustness_results/9_expectations_euclidean.csv', index=False)

In [72]:
df = pd.read_csv('odd_market_robustness_results/9_expectations_euclidean.csv')

self_manipulation_count = []
    
for i, row in df.iterrows():
    mpref = json.loads(row['mpref'])
    wpref = json.loads(row['wpref'])

    n = len(mpref)
    count = 0
    original_matching = gale_shapley(mpref, wpref)
    for manipulator in range(1, n + 1):
        manipulated_matching = self_manipulation(mpref, wpref, manipulator)
        if manipulated_matching[0] != original_matching:
            count += 1
    self_manipulation_count.append(count)

df['self_manipulating_women'] = self_manipulation_count
df.to_csv('odd_market_robustness_results/9_expectations_euclidean.csv', index=False)

In [73]:
def invert_proposer_matches(proposer_matches):
    """
    proposer_matches: list of length n where index i (0-based) is proposer (i+1)
                      and value proposer_matches[i] is the partner id (1-based).
    Returns: list `inv` of length n that maps partner -> proposer (1-based ids),
             i.e., inv[partner-1] = proposer_id
    """
    n = len(proposer_matches)
    inv = [0] * n
    for proposer_idx, partner in enumerate(proposer_matches):
        if partner == 0:
            # unmatched / sentinel (shouldn't happen in full matching)
            continue
        inv[partner - 1] = proposer_idx + 1
    return inv


def rank_differences_women(mpref, wpref, debug=False):
    """
    Return list of length n: for each woman w (1..n),
    diff[w-1] = rank_w(men-opt partner for w) - rank_w(women-opt partner for w).
    This should be >= 0 for all w (woman-optimal is at least as good).
    """
    # men-optimal: list mapping man -> woman (1-based IDs)
    men_optimal = gale_shapley(mpref, wpref)

    # women-proposing: returns list mapping woman -> man (1-based IDs)
    women_proposer_matches = gale_shapley(wpref, mpref)

    # invert to get men -> woman mapping for the women-optimal matching
    women_optimal = invert_proposer_matches(women_proposer_matches)

    n = len(mpref)
    differences = []

    for w in range(1, n + 1):
        # partner of woman w in men-optimal matching (this returns the man id who is matched to w)
        partner_men_opt = get_w_partner(w, men_optimal)   # returns man id
        # partner of woman w in women-optimal matching (man id)
        partner_women_opt = get_w_partner(w, women_optimal)

        # rank of those partners in woman's preference list (1-based ranks)
        rank_men_opt = wpref[w - 1].index(partner_men_opt) + 1
        rank_women_opt = wpref[w - 1].index(partner_women_opt) + 1

        diff = rank_men_opt - rank_women_opt
        differences.append(diff)

        if debug and diff < 0:
            print(f"DEBUG NEGATIVE: woman={w}, rank_men_opt={rank_men_opt}, "
                  f"rank_women_opt={rank_women_opt}, man_opt={partner_men_opt}, woman_opt={partner_women_opt}")
            print(" wpref[w-1] =", wpref[w-1])
            print(" men_optimal (man->woman) =", men_optimal)
            print(" women_proposer_matches (woman->man) =", women_proposer_matches)
            print(" women_optimal (man->woman) =", women_optimal)

    return differences


In [74]:
import pandas as pd
import ast
import numpy as np

# --- Your helper functions (from before) ---
# gale_shapley, get_w_partner, get_m_partner, rank_differences_women, etc.

def parse_preferences(pref_str):
    """Convert the string representation of preferences to lists."""
    return ast.literal_eval(pref_str)

def compute_differences(pref_str):
    """Compute rank differences for women for a given preference string."""
    mpref, wpref = parse_preferences(pref_str)
    return rank_differences_women(mpref, wpref)

def process_csv(filepath):
    """Load CSV, compute rank differences and average, return updated df."""
    df = pd.read_csv(filepath)
    
    # Identify the preferences column (assuming it's the 3rd column)
    df["preferences"] = df["mpref"] + "," + df["wpref"]  # Combine mpref and wpref into a single string
    pref_col = "preferences"  # Assuming the new column contains the preferences
    
    # Compute rank differences
    df["rank_differences"] = df[pref_col].apply(compute_differences)
    
    # Compute average rank difference per row
    df["avg_rank_differences"] = df["rank_differences"].apply(lambda x: np.mean(x))
    
    return df

def count_positive(lst_str):
    try:
        lst = ast.literal_eval(lst_str)  # convert string -> list
        return sum(1 for x in lst if x > 0)
    except (ValueError, SyntaxError):
        return 0  # fallback if parsing fails
    
def egalitarian_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n + 1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
        cost += wpref[w-1].index(m)
    return cost

import numpy as np


def find_all_stable_original(malechoice, femalechoice, n):
    # Output list with all matchings
    res = []
    # Initialization
    unchanged = [True] * n
    marriage = [0] * n
    malecounter = [0] * n
    # Female choice transformation

    fc = np.zeros((n, n), dtype=int)
    for i in range(1, n):
        for j in range(0, n - 1):
            fc[i][femalechoice[i][j]] = j
        fc[i][0] = n
        malecounter[i] = 0
        marriage[i] = 0

    def get_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        return(malematches)

    def print_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        print(*malematches)

    def proposal(i, malec, marriage, c):
        if i < 0:
            success = True
        elif (i == 0) or malec[i] == (n - 1) or not (unchanged[i]):
            success = False
        else:
            c += 1
            j = malec[i]
            malec[i] = j + 1
            success = refusal(i, malechoice[i][j], malec, marriage, c)
        return success

    def refusal(i, j, malec, marriage, c):
        if fc[j][abs(marriage[j])] > fc[j][i]:
            k = marriage[j]
            marriage[j] = i
            success = proposal(k, malec, marriage, c)
        else:
            success = proposal(i, malec, marriage, c)
        return success

    def breakmariage(malecounter, marriage, i, n, count):
        malecounter_ori = malecounter.copy()
        marriage_ori = marriage.copy()
        marriage[malechoice[i][malecounter[i] - 1]] = -i
        success = proposal(i, malecounter, marriage, count)

        if not (success):
            unchanged[i] = False
            malecounter_ref = malecounter_ori.copy()
            marriage_ref = marriage_ori.copy()
            malecounter = malecounter_ref
            marriage = marriage_ref
            return malecounter, marriage

        #print_marriage(marriage)  # marriage[i] means the woman who is married to i man
        new_marriage = get_marriage(marriage)  # convert into female solution
        res.append(new_marriage)
        # establish the relationships between the original and the obtained stable matching

        for j in range(i, n):
            # if i==2 and j==2:
            #     print("This is it")
            malecounter, marriage = breakmariage(malecounter, marriage, j, n, count)

        for j in range((i + 1), n):
            unchanged[j] = True
        # return malecounter_ori, abs(marriage_ori)
        return malecounter_ori, marriage_ori

    count = 0
    for i in range(1, n):
        proposal(i, malecounter, marriage, count)
    #print_marriage(marriage)

    new_marriage = get_marriage(marriage)  # convert into female solution
    res.append(new_marriage)
    # establish the relationships between the original and the obtained stable matching

    M_optimal_marriage = marriage.copy()
    M_optimal_counter = malecounter.copy()
    for i in range(1, n - 1):
        breakmariage(M_optimal_counter.copy(), M_optimal_marriage.copy(), i, n, count)
    return res

def men_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
    return cost

def women_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += wpref[w-1].index(m)
    return cost

In [75]:
files = [
    "odd_market_robustness_results/9_euclidean_mallows.csv",
    "odd_market_robustness_results/9_expectations_euclidean.csv",
    "odd_market_robustness_results/9_mallows.csv",
    "odd_market_robustness_results/9_mallows_MA.csv",
    "odd_market_robustness_results/9_mallows_MD.csv",
    "odd_market_robustness_results/9_reverse_euclidean.csv"
]
for f in files:
    df_updated = process_csv(f)
    out_file = f.replace(".csv", "_with_rank_diff.csv")
    df_updated.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")

rank_gain_files = [f.replace(".csv", "_with_rank_diff.csv") for f in files]

for f in rank_gain_files:
    df = pd.read_csv(f)
    df["trunc_count"] = df["rank_differences"].apply(count_positive)
    egalitarian_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = egalitarian_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        egalitarian_matches.append(best_match)
    df["egalitarian_match"] = egalitarian_matches
    men_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = men_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        men_matches.append(best_match)
    df["men_opt_match"] = men_matches

    women_matches = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref

        stable = find_all_stable_original(mpref, wpref, len(mpref))
        best_match = None
        best_cost = float("inf")
        for match in stable:
            cost = women_cost(match, m_pref, w_pref)
            if cost < best_cost:
                best_cost = cost
                best_match = match

        women_matches.append(best_match)
    df["women_opt_match"] = women_matches


    cost_diffs = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))

        egal = row["egalitarian_match"]  # <- no ast.literal_eval
        egal_cost = egalitarian_cost(egal, m_pref, w_pref)

        men_opt = row["men_opt_match"]
        men_cost1 = men_cost(men_opt, m_pref, w_pref)

        women_opt = row["women_opt_match"]
        women_cost1 = women_cost(women_opt, m_pref, w_pref)

        delta = egal_cost - men_cost1 - women_cost1
        cost_diffs.append(delta)



    df["trunc_egal"] = cost_diffs 

    stability_counts = []

    for idx in range(len(df)):
        row = df.loc[idx]
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        m_pref = [[int(x) for x in sublist] for sublist in mpref]
        w_pref = [[int(x) for x in sublist] for sublist in wpref]
        mpref = [[]] + m_pref
        wpref = [[]] + w_pref
        stable = find_all_stable_original(mpref, wpref, len(mpref))
        num_stable = len(stable)
        stability_counts.append(num_stable)


    df["stability_count"] = stability_counts

    df["trunc_prevalence"] = (df["stability_count"] > 1).astype(int)

    df.to_csv(f)

Saved: odd_market_robustness_results/9_euclidean_mallows_with_rank_diff.csv
Saved: odd_market_robustness_results/9_expectations_euclidean_with_rank_diff.csv
Saved: odd_market_robustness_results/9_mallows_with_rank_diff.csv
Saved: odd_market_robustness_results/9_mallows_MA_with_rank_diff.csv
Saved: odd_market_robustness_results/9_mallows_MD_with_rank_diff.csv
Saved: odd_market_robustness_results/9_reverse_euclidean_with_rank_diff.csv


# Two for One

These cells follow the size-4 workflow in `market_4_data.ipynb`. 
The run updates the existing `_with_rank_diff.csv` files for all three market sizes. 
It uses the original report selection and absolute rank-change measure for comparability. The reference search indexes the man-to-woman matching at the beneficiary’s ID when choosing reports. The exported scores then use the woman’s actual partner. An absolute rank change can include a worsening.

In [76]:
## Two for One Code

import itertools
import copy

def generate_sw_manipulations(w_pref):
    """
    Sw: Move any two distinct men to top of w's list in any order.
    """
    manipulations = []
    for mi, mj in itertools.permutations(w_pref, 2): 
        rest = [m for m in w_pref if m != mi and m != mj]
        new_pref = [mi, mj] + rest
        manipulations.append(new_pref)
    return manipulations


def generate_sm_manipulations(m_pref, mu_m):
    """
    Sm: Promote µ(m) to top, then for every other woman, promote her to top of that list.
    """
    sm = []
    bm = [mu_m] + [w for w in m_pref if w != mu_m]
    sm.append(bm)

    for w0 in bm[1:]:  # exclude mu(m) which is already at the top
        manipulated = [w0] + [w for w in bm if w != w0]
        sm.append(manipulated)

    return sm


def two_for_one(mpref, wpref, accomplice, beneficiary, initial_matching):
    #initial_matching = gale_shapley(mpref, wpref)
    mpref_prime = copy.deepcopy(mpref)
    wpref_prime = copy.deepcopy(wpref)
    best_matching = initial_matching

    accomplice_pref = mpref[accomplice - 1] 
    beneficiary_pref = wpref[beneficiary - 1]

    Sw = generate_sw_manipulations(beneficiary_pref)

    for w_misreport in Sw:
        #t5 = time.perf_counter()

        Sm = generate_sm_manipulations(accomplice_pref, initial_matching[accomplice - 1])

        for m_misreport in Sm:
            #t6 = time.perf_counter()
            mpref_prime[accomplice - 1] = m_misreport 
            wpref_prime[beneficiary - 1] = w_misreport
            new_matching = gale_shapley(mpref_prime, wpref_prime)

            new_w_pairing = new_matching[beneficiary - 1]
            new_m_pairing = new_matching[accomplice - 1]

            if beneficiary_pref.index(new_w_pairing) < beneficiary_pref.index(best_matching[beneficiary - 1]) and best_matching[accomplice - 1] == new_m_pairing:
                best_matching = new_matching
                accomplice_pref = m_misreport
                beneficiary_pref = w_misreport
        #t7 = time.perf_counter()
        #print(f"Time for t5 - t6: {t6 - t5:.4f} seconds")
        #print(f"Time for t6 - t7: {t7 - t6:.4f} seconds")
    return best_matching, accomplice_pref, beneficiary_pref

In [77]:
# The search above is the reference. Compile the inner trials for the full sweep.
import ast
from numba import njit

@njit
def trial_matching(mpref, wrank, trial, next_choice, women_partners):
    next_choice[:] = 0
    women_partners[:] = -1
    for proposer in range(len(mpref)):
        man = proposer
        while man >= 0:
            woman = mpref[man, next_choice[man]]
            next_choice[man] += 1
            incumbent = women_partners[woman]
            if incumbent < 0 or wrank[woman, man] < wrank[woman, incumbent]:
                women_partners[woman] = man
                trial[man] = woman
                man = incumbent

@njit
def search_reports(mpref, wpref, initial, women_reports, women_report_ranks):
    n = len(initial)
    wrank = np.empty_like(wpref)
    for w in range(n):
        for pos in range(n):
            wrank[w, wpref[w, pos]] = pos
    reported_men = mpref.copy()
    reported_wrank = wrank.copy()
    results = np.empty((n, n, 3, n), dtype=np.int64)
    trial = np.empty(n, dtype=np.int64)
    women_partners = np.empty_like(trial)
    next_choice = np.empty_like(trial)
    base_order = np.empty_like(trial)

    for beneficiary in range(n):
        for accomplice in range(n):
            best, accomplice_pref, beneficiary_pref = results[beneficiary, accomplice]
            best[:] = initial
            accomplice_pref[:] = mpref[accomplice]
            beneficiary_pref[:] = wpref[beneficiary]
            beneficiary_rank = wrank[beneficiary].copy()
            original_partner = initial[accomplice]
            for report_index in range(n * (n - 1)):
                w_report = women_reports[beneficiary, report_index]
                w_report_rank = women_report_ranks[beneficiary, report_index]
                reported_wrank[beneficiary] = w_report_rank
                # The reference rebuilds Sm from the latest accepted report.
                base_order[0] = original_partner
                j = 1
                for woman in accomplice_pref:
                    if woman != original_partner:
                        base_order[j] = woman
                        j += 1
                for promote in range(n):
                    men_row = reported_men[accomplice]
                    men_row[0] = base_order[promote]
                    men_row[1:promote + 1] = base_order[:promote]
                    men_row[promote + 1:] = base_order[promote + 1:]
                    trial_matching(reported_men, reported_wrank, trial, next_choice, women_partners)
                    # The index and comparison match market_4_data.ipynb.
                    if (beneficiary_rank[trial[beneficiary]] < beneficiary_rank[best[beneficiary]]
                            and trial[accomplice] == best[accomplice]):
                        best[:] = trial
                        accomplice_pref[:] = men_row
                        beneficiary_pref[:] = w_report
                        beneficiary_rank[:] = w_report_rank
            reported_men[accomplice] = mpref[accomplice]
        reported_wrank[beneficiary] = wrank[beneficiary]
    return results + 1

def two_for_one_outcomes(mpref, wpref, initial_matching):
    women_reports = np.asarray([generate_sw_manipulations(order) for order in wpref]) - 1
    return search_reports(np.asarray(mpref) - 1, np.asarray(wpref) - 1,
                          np.asarray(initial_matching) - 1, women_reports,
                          np.argsort(women_reports, axis=2)).tolist()

cultures = ("euclidean_mallows", "expectations_euclidean", "mallows",
            "mallows_MA", "mallows_MD", "reverse_euclidean")
two_for_one_files = [
    Path("odd_market_robustness_results") / f"{n}_{culture}_with_rank_diff.csv"
    for n in (5, 7, 9) for culture in cultures
]

for filepath in two_for_one_files:
    sample = pd.read_csv(filepath, usecols=["preferences"])
    for idx in (0, len(sample) // 2, len(sample) - 1):
        men, women = ast.literal_eval(sample.iloc[idx]["preferences"])
        initial = gale_shapley(men, women)
        outcomes = two_for_one_outcomes(men, women, initial)
        n = len(men)
        for w in range(n):
            for m in range(n):
                assert outcomes[w][m] == list(two_for_one(men, women, m + 1, w + 1, initial))
print("Matched the reference code on three profiles from every input CSV.")

Matched the reference code on three profiles from every input CSV.


In [78]:
for filepath in two_for_one_files:
    df = pd.read_csv(filepath)
    results = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        mpref, wpref = ast.literal_eval(row["preferences"]) 
        mpref = [[int(x) for x in sublist] for sublist in mpref]
        wpref = [[int(x) for x in sublist] for sublist in wpref]

        n = len(mpref)
        #original_matching = df["original matchings"][idx]

        original_matching = gale_shapley(mpref, wpref)
        outcomes = two_for_one_outcomes(mpref, wpref, original_matching)

        instance_results = []  # store all manipulator-accomplice outcomes for this row

        for manipulator in range(1, n+1):
            manipulator_results = []  # all accomplices for this manipulator

            for accomplice in range(1, n+1):
                func_return = outcomes[manipulator - 1][accomplice - 1]
                new_matching = func_return[0]
                accomplice_pref_after, beneficiary_pref_after = func_return[1:]

                # --- compute benefit ---
                rk1 = rk2 = 1000
                for i in range(n):
                    if original_matching[i] == manipulator:
                        partner1 = i + 1
                    if new_matching[i] == manipulator:
                        partner2 = i + 1
            
                w_pref = wpref[manipulator - 1]
                for i in range(n):
                    if w_pref[i] == partner1:
                        rk1 = i
                    if w_pref[i] == partner2:
                        rk2 = i

                benefit = abs(rk1 - rk2)  # absolute change in the truthful rank

                # store for this accomplice
                manipulator_results.append([
                    benefit,
                    new_matching,
                    accomplice_pref_after,
                    beneficiary_pref_after
                ])

            # store for this manipulator
            instance_results.append(manipulator_results)

        # store results for this row
        results.append(instance_results)

    #print(results)

    # attach to df
    df["two_for_one_results"] = results
    df.to_csv(filepath, index=False)
    print(f"Two-for-one completed: {filepath.name} ({len(df):,} profiles)", flush=True)

Two-for-one completed: 5_euclidean_mallows_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 5_expectations_euclidean_with_rank_diff.csv (1,500 profiles)
Two-for-one completed: 5_mallows_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 5_mallows_MA_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 5_mallows_MD_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 5_reverse_euclidean_with_rank_diff.csv (1,500 profiles)
Two-for-one completed: 7_euclidean_mallows_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 7_expectations_euclidean_with_rank_diff.csv (1,500 profiles)
Two-for-one completed: 7_mallows_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 7_mallows_MA_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 7_mallows_MD_with_rank_diff.csv (1,800 profiles)
Two-for-one completed: 7_reverse_euclidean_with_rank_diff.csv (1,500 profiles)
Two-for-one completed: 9_euclidean_mallows_with_rank_diff.csv (1,800 profiles)
Two-for-one

In [79]:
for filepath in two_for_one_files:
    df = pd.read_csv(filepath)
    # Prepare lists to store the new columns
    best_benefits_list = []
    avg_best_benefit_list = []
    max_best_benefit_list = []
    two_freq_instance_list = []
    number_two_pairs_list = []
    number_two_manipulators_list = []

    for idx, row in df.iterrows():
        # Convert string to list if needed
        two_for_one_results = row["two_for_one_results"]
        if isinstance(two_for_one_results, str):
            two_for_one_results = json.loads(two_for_one_results)

        # Best benefit per manipulator
        best_benefits_per_woman = [
            max(result[0] for result in manipulator_results)
            for manipulator_results in two_for_one_results
        ]
        best_benefits_list.append(best_benefits_per_woman)
        max_best_benefit_list.append(max(best_benefits_per_woman))

        # Average across manipulators
        avg_val = np.mean(best_benefits_per_woman)
        avg_best_benefit_list.append(avg_val)

        # --- New columns ---
        two_freq_instance_list.append(1 if avg_val > 0 else 0)

        # number_two_pairs: manipulator has at least one accomplice with benefit > 0
        #num_pairs = sum(
            #any(result[0] > 0 for result in manipulator_results)
            #for manipulator_results in two_for_one_results
        #)
        #number_two_pairs_list.append(num_pairs)
    
    
        num_pairs = sum(
        sum(1 for result in manipulator_results if result[0] > 0)
        for manipulator_results in two_for_one_results
        )
        number_two_pairs_list.append(num_pairs)

        # number_two_manipulators: count of >0 best benefits
        num_manipulators = sum(1 for b in best_benefits_per_woman if b > 0)
        number_two_manipulators_list.append(num_manipulators)

    # Add new columns to the original DataFrame
    df["best_benefits_per_woman_two"] = best_benefits_list
    df["max_best_benefit_two"] = max_best_benefit_list
    df["avg_best_benefit_two"] = avg_best_benefit_list
    df["two_freq_instance"] = two_freq_instance_list
    df["number_two_pairs"] = number_two_pairs_list
    df["number_two_manipulators"] = number_two_manipulators_list

    # --- Save the updated DataFrame to CSV ---
    df.to_csv(filepath, index=False)
    print(f"Two-for-one summaries saved: {filepath.name}")

Two-for-one summaries saved: 5_euclidean_mallows_with_rank_diff.csv
Two-for-one summaries saved: 5_expectations_euclidean_with_rank_diff.csv
Two-for-one summaries saved: 5_mallows_with_rank_diff.csv
Two-for-one summaries saved: 5_mallows_MA_with_rank_diff.csv
Two-for-one summaries saved: 5_mallows_MD_with_rank_diff.csv
Two-for-one summaries saved: 5_reverse_euclidean_with_rank_diff.csv
Two-for-one summaries saved: 7_euclidean_mallows_with_rank_diff.csv
Two-for-one summaries saved: 7_expectations_euclidean_with_rank_diff.csv
Two-for-one summaries saved: 7_mallows_with_rank_diff.csv
Two-for-one summaries saved: 7_mallows_MA_with_rank_diff.csv
Two-for-one summaries saved: 7_mallows_MD_with_rank_diff.csv
Two-for-one summaries saved: 7_reverse_euclidean_with_rank_diff.csv
Two-for-one summaries saved: 9_euclidean_mallows_with_rank_diff.csv
Two-for-one summaries saved: 9_expectations_euclidean_with_rank_diff.csv
Two-for-one summaries saved: 9_mallows_with_rank_diff.csv
Two-for-one summaries s